<a href="https://colab.research.google.com/github/MEY1337/DeepLeanning-project/blob/main/Titanic_Dataset_Load.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Titanic 생존 예측 프로젝트
# Colab 또는 Jupyter Notebook에서 실행 가능

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

!pip install kaggle

from google.colab import files
files.upload()  # kaggle.json 업로드

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c titanic
!unzip titanic.zip

# 1. 데이터 불러오기
# 파일 업로드 후 train.csv, test.csv 경로를 맞춰서 사용하세요.
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print("train shape:", train.shape)
print("test shape:", test.shape)
print(train.head())

# 2. 결측치 확인
print("\n[결측치 개수]")
print(train.isnull().sum())

# 3. 전처리 함수
def preprocess(df):
    df = df.copy()

    # Cabin은 결측치가 많아 제외
    if 'Cabin' in df.columns:
        df = df.drop(columns=['Cabin'])

    # Name, Ticket, PassengerId는 이번 기본 모델에서 제외
    drop_cols = []
    for col in ['Name', 'Ticket', 'PassengerId']:
        if col in df.columns:
            drop_cols.append(col)
    if drop_cols:
        df = df.drop(columns=drop_cols)

    # 파생변수 생성
    if 'SibSp' in df.columns and 'Parch' in df.columns:
        df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    # 결측치 처리
    if 'Age' in df.columns:
        df['Age'] = df['Age'].fillna(df['Age'].median())

    if 'Embarked' in df.columns:
        df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    if 'Fare' in df.columns:
        df['Fare'] = df['Fare'].fillna(df['Fare'].median())

    # 범주형 변수 원-핫 인코딩
    df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

    return df

# 4. 학습 데이터와 정답 분리
X = train.drop(columns=['Survived'])
y = train['Survived']

X = preprocess(X)
X_test_real = preprocess(test)

# train/test 컬럼 맞추기
X, X_test_real = X.align(X_test_real, join='left', axis=1, fill_value=0)

# 5. 검증용 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. 스케일링 (KNN, Logistic Regression용)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# 7. 모델 정의
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

# 8. 모델 학습 및 평가
results = []

for name, model in models.items():
    if name in ["Logistic Regression", "KNN"]:
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_valid_scaled)
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_valid)

    acc = accuracy_score(y_valid, pred)
    prec = precision_score(y_valid, pred)
    rec = recall_score(y_valid, pred)
    f1 = f1_score(y_valid, pred)

    results.append([name, acc, prec, rec, f1])

    print(f"\n[{name}]")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall   :", round(rec, 4))
    print("F1-score :", round(f1, 4))
    print(classification_report(y_valid, pred))

# 9. 결과 표
result_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1-score"])
print("\n[모델 비교 결과]")
print(result_df.sort_values(by="Accuracy", ascending=False))

# 10. 최종 모델 선택 후 test.csv 예측 예시
best_model = RandomForestClassifier(n_estimators=200, random_state=42)
best_model.fit(X, y)

test_pred = best_model.predict(X_test_real)

# 제출 파일 생성
submission = pd.read_csv('gender_submission.csv') if 'gender_submission.csv' else pd.DataFrame()
if not submission.empty and 'PassengerId' in submission.columns:
    submission['Survived'] = test_pred
else:
    # gender_submission.csv가 없다면 PassengerId를 test 원본에서 가져옴
    submission = pd.DataFrame({
        'PassengerId': test['PassengerId'],
        'Survived': test_pred
    })

submission.to_csv('submission.csv', index=False)
print("\nsubmission.csv 파일이 생성되었습니다.")
print(submission.head())

Saving titanic.zip to titanic.zip
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
Archive:  titanic.zip
  inflating: gender_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
train shape: (891, 12)
test shape: (418, 11)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                        